<a href="https://colab.research.google.com/github/behan02/langchain-tutorial/blob/main/chapters/08-conditional_chains.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Conditional Chains

In [1]:
!pip install -qU \
  langchain-core \
  langchain-google-genai \
  langchain-community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.8/68.8 kB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 23.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 28.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 2.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [2]:
from langchain_google_genai import ChatGoogleGenerativeAI
import os

os.environ["GOOGLE_API_KEY"] = ""

In [3]:
from pydantic import BaseModel
from typing import Literal

class llm_schema(BaseModel):
  movie_summary_flag: Literal["positive", "negative"]

In [4]:
from langchain_core.prompts import ChatPromptTemplate

prompt_template = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a movie review evaluator"),
        ("human", "Please categorize the movie review as positive or negative : {review}")
    ]
)

In [5]:
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash",temperature=0)

llm_structured_output = llm.with_structured_output(llm_schema)

In [6]:
from langchain_core.runnables import RunnableParallel, RunnableLambda, RunnableBranch

def pydantic_json(input:llm_schema) -> str:
  return input.model_dump()['movie_summary_flag']

pydantic_json_lambda = RunnableLambda(pydantic_json)

## Conditional Chain 1

In [7]:
from langchain_core.output_parsers import StrOutputParser

linkedin_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a LinkedIn post generator"),
    ("human", "Create a post for the following text for LinkedIn: {text}")])

str_parser = StrOutputParser()

chain_linkedin = linkedin_prompt | llm | str_parser

## Conditional Chain 2

In [14]:
def insta_chain(text:str):

    insta_prompt = ChatPromptTemplate.from_messages([
      ("system", "You are a LinkedIn post generator"),
      ("human", "Create a post for the following text for Instagram: {text}")])

    chain_insta = insta_prompt | llm | str_parser

    result = chain_insta.invoke(text)

    return result

insta_chain_runnable = RunnableLambda(insta_chain)

## Final chain

### Understanding the Logic

To create the conditional routing in our chain, we use two key concepts:

1. **`RunnableBranch`**: This is a LangChain primitive used to define routing logic. It takes a list of (condition, runnable) pairs. It evaluates the conditions in order and executes the first runnable whose condition evaluates to true. If no conditions match, it executes a default runnable provided as the last argument.

2. **`lambda` functions**: These are small, anonymous 'one-liner' functions in Python. In our case, `lambda x: "positive" in x` is a quick way to define a function that checks if the string 'positive' exists in the input coming from the previous step of the chain.

In [15]:
conditional_chain = RunnableBranch(
    (lambda x: "positive" in x, chain_linkedin),
     insta_chain_runnable
)

final_orchestrator = prompt_template | llm_structured_output | pydantic_json_lambda | conditional_chain

In [17]:
final_orchestrator.invoke({"review": "I like Inception movie"})

'Okay, here are a few options for a LinkedIn post based on the word "positive," each with a slightly different angle. Choose the one that best fits your personal brand or the message you want to convey!\n\n---\n\n**Option 1: Focusing on Mindset & Impact**\n\nThe power of **positive** thinking isn\'t just a feel-good phrase; it\'s a strategic asset. ✨\n\nA positive mindset fuels resilience, sparks innovation, and builds stronger teams. Even on challenging days, choosing to approach situations with optimism can transform obstacles into opportunities.\n\nHow do you cultivate positivity in your professional life? Share your tips below!\n\n#Positivity #Mindset #Leadership #WorkplaceCulture #ProfessionalGrowth\n\n---\n\n**Option 2: Focusing on Action & Ripple Effect**\n\nIt\'s amazing how a single word or action can shift an entire day, a meeting, or even a project. Today, let\'s focus on being **positive**. 🌱\n\nA genuine compliment, a word of encouragement, a solution-oriented approach – t